# SMAP dryness trigger vs USDM DSCI ≥ 200: corrected lead-window analysis

This notebook checks the lead-window results in the JARS manuscript. Run it with **Runtime → Run all**.

It compares three versions for each lead window:

| Version | Event labels | Trigger |
|---|---|---|
| **A_original** | the original notebook method (the current week plus the next L weekly map rows) | original trigger (last 5 valid days) |
| **B_fixed_label** | DSCI ≥ 200 on any USDM map date in (t, t+L] days, matching Eq. (7) | original trigger |
| **C_fixed_both** | same as B | trigger counted over calendar days t−4 to t, matching Eq. (6) |

**Version C matches the Methods text.** Everything is saved to a new Drive folder: `SMAP_preprocessed/Corrected_LeadWindow_Results/`.

This notebook was originally run in Google Colab (https://colab.research.google.com/drive/1K7YAipuD8_n3t94Gfo3uy2r2OVlyN-6D).

In [ ]:
# 1. Setup: mount Drive, set paths, create the output folder
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

out_base    = "/content/drive/MyDrive/SMAP_preprocessed"
usdm_path   = os.path.join(out_base, "USDM", "USDM_county_2016_2020.csv.csv")
county_path = os.path.join(out_base, "SMAP_county_daily_2016_2020_with_anom.parquet")

results_dir = os.path.join(out_base, "Corrected_LeadWindow_Results")
os.makedirs(results_dir, exist_ok=True)
print("Results will be saved in:", results_dir)

lead_times  = [7, 14, 21, 30, 45, 60]
DSCI_THRESH = 200

In [ ]:
# 2. Daily county SMAP series + calendar-day trigger (Eq. 6)
county = pd.read_parquet(county_path)
county['date'] = pd.to_datetime(county['date'])
county['county_fips'] = county['county_fips'].astype(str).str.zfill(5)
county = county.sort_values(['county_fips', 'date']).reset_index(drop=True)

# Original trigger (as in your notebook): >= 3 dry values among the last 5 available rows
county['dry'] = (county['sm_z'] <= -1.0).astype(int)
if 'trigger' not in county.columns:
    county['trigger'] = (county.groupby('county_fips')['dry']
                               .transform(lambda s: s.rolling(5, min_periods=1).sum()) >= 3).astype(int)

# Corrected trigger: >= 3 dry days among calendar days t-4..t
trig_5d = []
for _, g in county.groupby('county_fips', sort=True):
    s = g.set_index('date')['dry'].rolling('5D').sum()   # window (t-5 days, t] = t-4..t
    trig_5d.append((s.values >= 3).astype(int))
county['trigger_5D'] = np.concatenate(trig_5d)

print("County-days:", len(county), "| counties:", county['county_fips'].nunique())
print("Trigger active (original): %.3f | (calendar-day): %.3f"
      % (county['trigger'].mean(), county['trigger_5D'].mean()))

In [ ]:
# 3. USDM weekly DSCI (full record) and verification pairs
usdm = pd.read_csv(usdm_path)
usdm.columns = usdm.columns.str.strip()
usdm['county_fips'] = usdm['FIPS'].astype(str).str.zfill(5)
usdm['date'] = pd.to_datetime(usdm['MapDate'].astype(str), format="%Y%m%d")
usdm = usdm[['county_fips', 'date', 'DSCI']]
usdm = usdm[usdm['county_fips'].isin(county['county_fips'].unique())]
usdm = usdm.sort_values(['county_fips', 'date']).reset_index(drop=True)
usdm['event'] = (usdm['DSCI'] >= DSCI_THRESH).astype(int)
last_map_date = usdm['date'].max()

# Verification pairs = county x USDM map date with valid SMAP
pairs = county.merge(usdm, on=['county_fips', 'date'], how='inner')
pairs = pairs.sort_values(['county_fips', 'date']).reset_index(drop=True)
print("Verification pairs:", len(pairs), "| counties:", pairs['county_fips'].nunique())
print("Last USDM map date:", last_map_date.date())

In [ ]:
# 4. Future event labels
groups = pairs.groupby('county_fips', sort=False).indices

# 4a. ORIGINAL (reproduces your notebook): current row + next L rows
for L in lead_times:
    out = np.zeros(len(pairs), dtype=int)
    ev = pairs['event'].values
    for _, idx in groups.items():
        y = ev[idx]; n = len(y)
        out[idx] = [int(y[i:min(n, i + L + 1)].any()) for i in range(n)]
    pairs[f'lab_orig_{L}'] = out

# 4b. FIXED (Eq. 7): any DSCI >= 200 on a USDM map date in (t, t+L] days.
# Uses the full USDM record; windows that run past the last map date are excluded (NaN).
usdm_groups = {k: g for k, g in usdm.groupby('county_fips', sort=False)}
for L in lead_times:
    lab = np.full(len(pairs), np.nan)
    for fips, idx in groups.items():
        g  = usdm_groups[fips]
        d  = g['date'].values
        cs = np.concatenate([[0], np.cumsum(g['event'].values)])
        t  = pairs['date'].values[idx]
        lo = np.searchsorted(d, t, side='right')
        hi = np.searchsorted(d, t + np.timedelta64(L, 'D'), side='right')
        vals = ((cs[hi] - cs[lo]) > 0).astype(float)
        vals[(t + np.timedelta64(L, 'D')) > np.datetime64(last_map_date)] = np.nan
        lab[idx] = vals
    pairs[f'lab_fix_{L}'] = lab
print("Labels done.")

In [ ]:
# 5. Verification scores -> CSV tables
def score(y, f):
    p  = y.mean()
    BS = brier_score_loss(y, f)
    return dict(N=len(y), event_rate=p,
                ROC_AUC=roc_auc_score(y, f), PR_AUC=average_precision_score(y, f),
                BS=BS, BSS=1 - BS / (p * (1 - p)),
                POD=f[y == 1].mean(), POFD=f[y == 0].mean())

versions = [('A_original',    'lab_orig', 'trigger'),
            ('B_fixed_label', 'lab_fix',  'trigger'),
            ('C_fixed_both',  'lab_fix',  'trigger_5D')]
rows = []
for L in lead_times:
    for name, lab, trig in versions:
        m = pairs[f'{lab}_{L}'].notna()
        y = pairs.loc[m, f'{lab}_{L}'].astype(int).values
        f = pairs.loc[m, trig].astype(float).values
        rows.append({'lead_days': L, 'version': name, **score(y, f)})
res = pd.DataFrame(rows)

res.to_csv(os.path.join(results_dir, "All_versions_verification.csv"), index=False)
table1 = (res[res['version'] == 'C_fixed_both']
          .drop(columns='version').round(3))
table1.to_csv(os.path.join(results_dir, "Table1_corrected.csv"), index=False)

pd.set_option('display.width', 200)
print("=== ALL VERSIONS ===")
print(res.round(4).to_string(index=False))
print("\n=== TABLE 1 (corrected, version C) ===")
print(table1.to_string(index=False))

In [ ]:
# 6. Figures (PNG, 300 dpi) saved to the results folder
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
style = {'A_original':    dict(color='#2a78d6', marker='o', label='A: original'),
         'B_fixed_label': dict(color='#eb6834', marker='s', label='B: corrected labels'),
         'C_fixed_both':  dict(color='#1baf7a', marker='^', label='C: corrected labels + trigger')}

# Figure A: comparison of all three versions (one metric per panel)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for name, st in style.items():
    r = res[res['version'] == name]
    axes[0].plot(r['lead_days'], r['ROC_AUC'], lw=2, ms=6, **st)
    axes[1].plot(r['lead_days'], r['PR_AUC'],  lw=2, ms=6, **st)
    axes[1].plot(r['lead_days'], r['event_rate'], lw=1.2, ls='--', color=st['color'])
    axes[2].plot(r['lead_days'], r['BSS'],     lw=2, ms=6, **st)
axes[0].axhline(0.5, color='0.6', lw=1, ls=':'); axes[0].set_ylabel('ROC–AUC')
axes[1].set_ylabel('PR–AUC (dashed = event rate)')
axes[2].axhline(0, color='0.6', lw=1, ls=':');   axes[2].set_ylabel('Brier skill score')
for ax, t in zip(axes, ['(a)', '(b)', '(c)']):
    ax.set_xlabel('Lead window (days)'); ax.set_xticks(lead_times)
    ax.set_title(t, loc='left'); ax.grid(axis='y', color='0.9')
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(results_dir, "FigA_versions_comparison.png"), dpi=300)
plt.show()

# Figure B: corrected results only (version C), paper-ready
r = res[res['version'] == 'C_fixed_both']
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].plot(r['lead_days'], r['ROC_AUC'], color='#2a78d6', marker='o', lw=2, ms=6, label='ROC–AUC')
axes[0].axhline(0.5, color='0.6', lw=1, ls=':', label='No-skill (0.5)')
axes[0].set_ylabel('ROC–AUC')
axes[1].plot(r['lead_days'], r['PR_AUC'], color='#2a78d6', marker='o', lw=2, ms=6, label='PR–AUC')
axes[1].plot(r['lead_days'], r['event_rate'], color='#eb6834', marker='s', lw=1.5, ms=6, ls='--',
             label='Event-rate baseline')
axes[1].set_ylabel('PR–AUC')
for ax, t in zip(axes, ['(a)', '(b)']):
    ax.set_xlabel('Lead window (days)'); ax.set_xticks(lead_times)
    ax.set_title(t, loc='left'); ax.grid(axis='y', color='0.9'); ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(results_dir, "FigB_corrected_skill_vs_lead.png"), dpi=300)
plt.show()

In [ ]:
# 7. Save the labeled verification table (for the Code and Data Availability record)
keep = ['county_fips', 'date', 'sm_z', 'trigger', 'trigger_5D', 'DSCI', 'event'] + \
       [c for c in pairs.columns if c.startswith(('lab_orig_', 'lab_fix_'))]
pairs[keep].to_parquet(os.path.join(results_dir, "verification_pairs_labeled.parquet"), index=False)

print("Saved files in", results_dir)
for f in sorted(os.listdir(results_dir)):
    print("  ", f)

In [ ]:
# ================================================================
# 8. ADDITIONAL EARLY-WARNING ANALYSES
#    Corrected labels + continuous SMAP + onset-only evaluation
# ================================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# ---------------------------------------------------------------
# 8.1 Continuous SMAP dryness score
# ---------------------------------------------------------------
# More negative soil-moisture anomalies indicate greater dryness.
# Multiplying by -1 makes larger values = stronger drought signal,
# which is the appropriate direction for ROC/PR calculations.

pairs["neg_z"] = -pairs["sm_z"]


# ---------------------------------------------------------------
# 8.2 Verification function
# ---------------------------------------------------------------

def verify_extended(df, score_col, onset_only=False, binary=False):

    rows = []

    for L in lead_times:

        label_col = f"lab_fix_{L}"

        # Keep observations with both outcome and forecast score
        d = df.dropna(subset=[label_col, score_col]).copy()

        # -------------------------------------------------------
        # ONSET-ONLY CONDITION
        # -------------------------------------------------------
        # event == 0 means DSCI < 200 at forecast initialization.
        # Therefore, counties already at/above DSCI 200 are removed.
        if onset_only:
            d = d[d["event"] == 0].copy()

        if len(d) == 0:
            continue

        y = d[label_col].astype(int).values
        f = d[score_col].astype(float).values

        # Need both event and non-event cases
        if np.unique(y).size < 2:
            continue

        p = y.mean()

        row = {
            "lead_days": L,
            "N": len(y),
            "event_rate": p,
            "ROC_AUC": roc_auc_score(y, f),
            "PR_AUC": average_precision_score(y, f)
        }

        # -------------------------------------------------------
        # Binary trigger metrics
        # -------------------------------------------------------
        if binary:

            bs = brier_score_loss(y, f)

            bs_ref = p * (1 - p)

            row["BS"] = bs

            row["BSS"] = (
                1 - bs / bs_ref
                if bs_ref > 0
                else np.nan
            )

            # Probability of detection
            row["POD"] = (
                f[y == 1].mean()
                if np.any(y == 1)
                else np.nan
            )

            # Probability of false detection
            row["POFD"] = (
                f[y == 0].mean()
                if np.any(y == 0)
                else np.nan
            )

            # Precision / positive predictive value
            predicted_positive = f == 1

            row["Precision"] = (
                y[predicted_positive].mean()
                if predicted_positive.sum() > 0
                else np.nan
            )

            # Critical Success Index
            TP = np.sum((f == 1) & (y == 1))
            FP = np.sum((f == 1) & (y == 0))
            FN = np.sum((f == 0) & (y == 1))

            denom = TP + FP + FN

            row["CSI"] = (
                TP / denom
                if denom > 0
                else np.nan
            )

        rows.append(row)

    return pd.DataFrame(rows)


# ================================================================
# 8.3 RUN FOUR ANALYSES
# ================================================================

# ---------------------------------------------------------------
# A. Current manuscript analysis
# Corrected calendar-day binary trigger
# Includes persistence + recurrence + development
# ---------------------------------------------------------------

binary_all = verify_extended(
    pairs,
    score_col="trigger_5D",
    onset_only=False,
    binary=True
)


# ---------------------------------------------------------------
# B. Continuous SMAP score
# Includes persistence + recurrence + development
# ---------------------------------------------------------------

continuous_all = verify_extended(
    pairs,
    score_col="neg_z",
    onset_only=False,
    binary=False
)


# ---------------------------------------------------------------
# C. Binary trigger, onset-only
# Counties already DSCI >= 200 at initialization are excluded
# ---------------------------------------------------------------

binary_onset = verify_extended(
    pairs,
    score_col="trigger_5D",
    onset_only=True,
    binary=True
)


# ---------------------------------------------------------------
# D. Continuous SMAP score, onset-only
# THIS IS THE MOST IMPORTANT NEW ANALYSIS
# ---------------------------------------------------------------

continuous_onset = verify_extended(
    pairs,
    score_col="neg_z",
    onset_only=True,
    binary=False
)


# ================================================================
# 8.4 DISPLAY RESULTS
# ================================================================

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

print("\n" + "=" * 80)
print("A. BINARY TRIGGER — ALL CASES (CURRENT PAPER)")
print("=" * 80)
print(binary_all.round(4).to_string(index=False))


print("\n" + "=" * 80)
print("B. CONTINUOUS SMAP SCORE — ALL CASES")
print("=" * 80)
print(continuous_all.round(4).to_string(index=False))


print("\n" + "=" * 80)
print("C. BINARY TRIGGER — ONSET ONLY")
print("=" * 80)
print(binary_onset.round(4).to_string(index=False))


print("\n" + "=" * 80)
print("D. CONTINUOUS SMAP SCORE — ONSET ONLY")
print("=" * 80)
print(continuous_onset.round(4).to_string(index=False))


# ================================================================
# 8.5 SAVE TABLES
# ================================================================

binary_all.to_csv(
    os.path.join(
        results_dir,
        "Sensitivity_binary_all.csv"
    ),
    index=False
)

continuous_all.to_csv(
    os.path.join(
        results_dir,
        "Sensitivity_continuous_all.csv"
    ),
    index=False
)

binary_onset.to_csv(
    os.path.join(
        results_dir,
        "Sensitivity_binary_onset.csv"
    ),
    index=False
)

continuous_onset.to_csv(
    os.path.join(
        results_dir,
        "Sensitivity_continuous_onset.csv"
    ),
    index=False
)


# ================================================================
# 8.6 COMPACT COMPARISON TABLE
# ================================================================

comparison = pd.DataFrame({
    "lead_days": lead_times,

    "Binary_all_AUC":
        binary_all["ROC_AUC"].values,

    "Continuous_all_AUC":
        continuous_all["ROC_AUC"].values,

    "Binary_onset_AUC":
        binary_onset["ROC_AUC"].values,

    "Continuous_onset_AUC":
        continuous_onset["ROC_AUC"].values,

    "Binary_all_PR":
        binary_all["PR_AUC"].values,

    "Continuous_all_PR":
        continuous_all["PR_AUC"].values,

    "Binary_onset_PR":
        binary_onset["PR_AUC"].values,

    "Continuous_onset_PR":
        continuous_onset["PR_AUC"].values
})


comparison.to_csv(
    os.path.join(
        results_dir,
        "Sensitivity_AUC_PR_comparison.csv"
    ),
    index=False
)


print("\n" + "=" * 80)
print("AUC / PR-AUC COMPARISON")
print("=" * 80)

print(
    comparison.round(4).to_string(index=False)
)


print(
    "\nSaved sensitivity-analysis files to:\n",
    results_dir
)

In [ ]:
# ======================================================================
# 9. STRICT DROUGHT-ONSET + CONTINUOUS SMAP + UNCERTAINTY ANALYSIS
#
# Add AFTER Cell [8] in the corrected notebook.
#
# Requires objects already created:
#   pairs
#   usdm
#   lead_times
#   results_dir
#
# Primary strict-onset definition:
#   DSCI < 200 at forecast initialization AND
#   no DSCI >= 200 during the previous 28 calendar days.
#
# Sensitivity:
#   14-, 28-, and 42-day antecedent drought-free periods.
#
# Forecast scores:
#   1) binary 5-day trigger
#   2) continuous SMAP dryness score = -sm_z
#
# Outputs:
#   - strict-onset verification tables
#   - onset sensitivity tables
#   - county-cluster bootstrap 95% CIs
#   - lead-time skill comparison
#   - publication-ready figures
# ======================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

# ----------------------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------------------

DSCI_THRESHOLD = 200

# Main definition
PRIMARY_ANTECEDENT_DAYS = 28

# Sensitivity analysis
ANTECEDENT_WINDOWS = [14, 28, 42]

# Bootstrap settings
N_BOOT = 1000
RANDOM_SEED = 2026

# Optional: use fewer iterations while testing the code
# N_BOOT = 200

strict_dir = os.path.join(
    results_dir,
    "Strict_Onset_Analysis"
)

os.makedirs(strict_dir, exist_ok=True)

print("Strict-onset outputs will be saved to:")
print(strict_dir)


# ======================================================================
# 9.1 PREPARE CONTINUOUS SMAP SCORE
# ======================================================================

# Larger score = drier soil = greater drought risk
pairs["neg_z"] = -pairs["sm_z"]

print("\nContinuous SMAP score created:")
print(pairs["neg_z"].describe())


# ======================================================================
# 9.2 CONSTRUCT STRICT ANTECEDENT DROUGHT-FREE FLAGS
# ======================================================================
#
# For every county x USDM map date:
#
# antecedent_clear_28 = 1 when there was NO DSCI >= 200
# during the previous 28 calendar days.
#
# IMPORTANT:
# Current date is excluded from the antecedent period.
# Current event status is checked separately.
#
# ======================================================================

usdm2 = usdm.copy()

usdm2["county_fips"] = (
    usdm2["county_fips"]
    .astype(str)
    .str.zfill(5)
)

usdm2["date"] = pd.to_datetime(usdm2["date"])

if "event" not in usdm2.columns:
    usdm2["event"] = (
        usdm2["DSCI"] >= DSCI_THRESHOLD
    ).astype(int)

usdm2 = (
    usdm2
    .sort_values(["county_fips", "date"])
    .reset_index(drop=True)
)


def add_antecedent_clear_flags(usdm_df, windows):

    output_parts = []

    for fips, g in usdm_df.groupby(
        "county_fips",
        sort=False
    ):

        g = g.copy().sort_values("date")

        dates = g["date"].values.astype(
            "datetime64[ns]"
        )

        event = g["event"].values.astype(int)

        # cumulative number of drought-event maps
        cumulative = np.concatenate(
            [[0], np.cumsum(event)]
        )

        for days in windows:

            flag = np.zeros(
                len(g),
                dtype=int
            )

            for i, t in enumerate(dates):

                start = (
                    t -
                    np.timedelta64(days, "D")
                )

                # Previous maps:
                # [t-days, t)
                lo = np.searchsorted(
                    dates,
                    start,
                    side="left"
                )

                hi = np.searchsorted(
                    dates,
                    t,
                    side="left"
                )

                previous_events = (
                    cumulative[hi] -
                    cumulative[lo]
                )

                flag[i] = int(
                    previous_events == 0
                )

            g[
                f"antecedent_clear_{days}d"
            ] = flag

        output_parts.append(g)

    return pd.concat(
        output_parts,
        ignore_index=True
    )


usdm_strict = add_antecedent_clear_flags(
    usdm2,
    ANTECEDENT_WINDOWS
)

keep_flags = [
    "county_fips",
    "date"
] + [
    f"antecedent_clear_{d}d"
    for d in ANTECEDENT_WINDOWS
]

pairs = pairs.merge(
    usdm_strict[keep_flags],
    on=["county_fips", "date"],
    how="left"
)

print("\nAntecedent drought-free fractions:")

for days in ANTECEDENT_WINDOWS:

    col = f"antecedent_clear_{days}d"

    print(
        f"{days:2d} days:",
        round(pairs[col].mean(), 4)
    )


# ======================================================================
# 9.3 DEFINE STRICT-ONSET ELIGIBILITY
# ======================================================================
#
# Eligible forecast initialization:
#
# 1. DSCI < 200 today
# 2. no DSCI >= 200 in preceding N days
#
# Future label remains:
#
# DSCI >= 200 at any map date in (t, t+L]
#
# ======================================================================

for days in ANTECEDENT_WINDOWS:

    pairs[
        f"strict_eligible_{days}d"
    ] = (
        (pairs["event"] == 0) &
        (
            pairs[
                f"antecedent_clear_{days}d"
            ] == 1
        )
    ).astype(int)


print("\nStrict-onset eligible verification dates:")

for days in ANTECEDENT_WINDOWS:

    col = f"strict_eligible_{days}d"

    print(
        f"{days:2d} days:",
        f"{pairs[col].sum():,}"
    )


# ======================================================================
# 9.4 VERIFICATION FUNCTION
# ======================================================================

def verify_strict(
    df,
    score_col,
    antecedent_days=28
):

    rows = []

    eligibility = (
        f"strict_eligible_"
        f"{antecedent_days}d"
    )

    for L in lead_times:

        label_col = f"lab_fix_{L}"

        d = df[
            df[eligibility] == 1
        ].dropna(
            subset=[
                label_col,
                score_col
            ]
        ).copy()

        y = (
            d[label_col]
            .astype(int)
            .values
        )

        score = (
            d[score_col]
            .astype(float)
            .values
        )

        if (
            len(y) == 0 or
            np.unique(y).size < 2
        ):
            continue

        prevalence = y.mean()

        auc = roc_auc_score(
            y,
            score
        )

        pr_auc = average_precision_score(
            y,
            score
        )

        rows.append({

            "antecedent_days":
                antecedent_days,

            "lead_days":
                L,

            "N":
                len(y),

            "events":
                int(y.sum()),

            "event_rate":
                prevalence,

            "ROC_AUC":
                auc,

            "PR_AUC":
                pr_auc,

            # PR-AUC relative to prevalence
            "PR_gain":
                (
                    pr_auc / prevalence
                    if prevalence > 0
                    else np.nan
                )
        })

    return pd.DataFrame(rows)


# ======================================================================
# 9.5 STRICT ONSET — CONTINUOUS SMAP
# ======================================================================

strict_continuous = {}

for days in ANTECEDENT_WINDOWS:

    strict_continuous[days] = (
        verify_strict(
            pairs,
            score_col="neg_z",
            antecedent_days=days
        )
    )

strict_continuous_all = pd.concat(
    strict_continuous.values(),
    ignore_index=True
)

strict_continuous_all.to_csv(
    os.path.join(
        strict_dir,
        "Strict_onset_continuous_SMAP.csv"
    ),
    index=False
)

print(
    "\n=========================================="
)
print(
    "STRICT ONSET — CONTINUOUS SMAP"
)
print(
    "=========================================="
)

print(
    strict_continuous_all
    .round(4)
    .to_string(index=False)
)


# ======================================================================
# 9.6 STRICT ONSET — BINARY TRIGGER
# ======================================================================

strict_binary = {}

for days in ANTECEDENT_WINDOWS:

    strict_binary[days] = (
        verify_strict(
            pairs,
            score_col="trigger_5D",
            antecedent_days=days
        )
    )

strict_binary_all = pd.concat(
    strict_binary.values(),
    ignore_index=True
)

strict_binary_all.to_csv(
    os.path.join(
        strict_dir,
        "Strict_onset_binary_trigger.csv"
    ),
    index=False
)

print(
    "\n=========================================="
)
print(
    "STRICT ONSET — BINARY TRIGGER"
)
print(
    "=========================================="
)

print(
    strict_binary_all
    .round(4)
    .to_string(index=False)
)


# ======================================================================
# 9.7 DIRECT CONTINUOUS-vs-BINARY COMPARISON
# ======================================================================

comparison = strict_continuous_all.merge(

    strict_binary_all,

    on=[
        "antecedent_days",
        "lead_days"
    ],

    suffixes=(
        "_continuous",
        "_binary"
    )
)

comparison[
    "AUC_improvement"
] = (
    comparison[
        "ROC_AUC_continuous"
    ] -
    comparison[
        "ROC_AUC_binary"
    ]
)

comparison[
    "PR_improvement"
] = (
    comparison[
        "PR_AUC_continuous"
    ] -
    comparison[
        "PR_AUC_binary"
    ]
)

comparison.to_csv(
    os.path.join(
        strict_dir,
        "Strict_onset_continuous_vs_binary.csv"
    ),
    index=False
)

print(
    "\n=========================================="
)
print(
    "CONTINUOUS vs BINARY"
)
print(
    "=========================================="
)

cols = [
    "antecedent_days",
    "lead_days",
    "event_rate_continuous",
    "ROC_AUC_binary",
    "ROC_AUC_continuous",
    "AUC_improvement",
    "PR_AUC_binary",
    "PR_AUC_continuous",
    "PR_improvement"
]

print(
    comparison[cols]
    .round(4)
    .to_string(index=False)
)


# ======================================================================
# 9.8 COUNTY-CLUSTER BOOTSTRAP
#
# Resample counties, not individual county-date rows.
#
# This preserves the temporal sequence within sampled counties.
#
# Primary analysis = 28-day antecedent period.
# ======================================================================

def county_cluster_bootstrap(
    df,
    score_col,
    antecedent_days=28,
    n_boot=1000,
    seed=2026
):

    rng = np.random.default_rng(seed)

    eligibility = (
        f"strict_eligible_"
        f"{antecedent_days}d"
    )

    all_results = []

    for L in lead_times:

        label_col = f"lab_fix_{L}"

        d = df[
            df[eligibility] == 1
        ].dropna(
            subset=[
                label_col,
                score_col
            ]
        ).copy()

        counties = (
            d["county_fips"]
            .unique()
        )

        grouped = {
            fips: g[
                [label_col, score_col]
            ].copy()
            for fips, g
            in d.groupby("county_fips")
        }

        boot_auc = []
        boot_pr = []

        for b in range(n_boot):

            sampled_counties = (
                rng.choice(
                    counties,
                    size=len(counties),
                    replace=True
                )
            )

            y_parts = []
            score_parts = []

            for fips in sampled_counties:

                g = grouped[fips]

                y_parts.append(
                    g[label_col].values
                )

                score_parts.append(
                    g[score_col].values
                )

            y = np.concatenate(y_parts)
            score = np.concatenate(
                score_parts
            )

            if np.unique(y).size < 2:
                continue

            boot_auc.append(
                roc_auc_score(
                    y,
                    score
                )
            )

            boot_pr.append(
                average_precision_score(
                    y,
                    score
                )
            )

        # Original estimate
        y0 = (
            d[label_col]
            .astype(int)
            .values
        )

        score0 = (
            d[score_col]
            .astype(float)
            .values
        )

        auc0 = roc_auc_score(
            y0,
            score0
        )

        pr0 = average_precision_score(
            y0,
            score0
        )

        all_results.append({

            "antecedent_days":
                antecedent_days,

            "lead_days":
                L,

            "N":
                len(d),

            "counties":
                len(counties),

            "event_rate":
                y0.mean(),

            "ROC_AUC":
                auc0,

            "ROC_AUC_CI_low":
                np.percentile(
                    boot_auc,
                    2.5
                ),

            "ROC_AUC_CI_high":
                np.percentile(
                    boot_auc,
                    97.5
                ),

            "PR_AUC":
                pr0,

            "PR_AUC_CI_low":
                np.percentile(
                    boot_pr,
                    2.5
                ),

            "PR_AUC_CI_high":
                np.percentile(
                    boot_pr,
                    97.5
                ),

            "bootstrap_iterations":
                len(boot_auc)
        })

        print(
            f"Finished {L:2d}-day lead "
            f"({len(boot_auc)} bootstrap samples)"
        )

    return pd.DataFrame(
        all_results
    )


print(
    "\n=========================================="
)
print(
    "RUNNING COUNTY-CLUSTER BOOTSTRAP"
)
print(
    "Primary definition: 28-day antecedent period"
)
print(
    "=========================================="
)

bootstrap_28 = county_cluster_bootstrap(

    pairs,

    score_col="neg_z",

    antecedent_days=
        PRIMARY_ANTECEDENT_DAYS,

    n_boot=N_BOOT,

    seed=RANDOM_SEED
)

bootstrap_28.to_csv(
    os.path.join(
        strict_dir,
        "Strict_onset_28d_bootstrap_CI.csv"
    ),
    index=False
)

print(
    "\n=========================================="
)
print(
    "28-DAY STRICT ONSET + 95% CI"
)
print(
    "=========================================="
)

print(
    bootstrap_28
    .round(4)
    .to_string(index=False)
)


# ======================================================================
# 9.9 PUBLICATION-READY AUC LEAD-TIME FIGURE
# ======================================================================

r = bootstrap_28.copy()

plt.figure(
    figsize=(7.5, 5.0)
)

yerr = np.vstack([

    r["ROC_AUC"] -
    r["ROC_AUC_CI_low"],

    r["ROC_AUC_CI_high"] -
    r["ROC_AUC"]

])

plt.errorbar(
    r["lead_days"],
    r["ROC_AUC"],
    yerr=yerr,
    marker="o",
    linewidth=2,
    capsize=4
)

plt.axhline(
    0.5,
    linestyle=":",
    linewidth=1
)

plt.xlabel(
    "Lead window (days)"
)

plt.ylabel(
    "ROC–AUC"
)

plt.title(
    "SMAP discrimination of subsequent drought development\n"
    "28-day antecedent drought-free requirement"
)

plt.xticks(
    lead_times
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        strict_dir,
        "Fig_StrictOnset_ROC_AUC_28d.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ======================================================================
# 9.10 PR-AUC vs EVENT-RATE BASELINE
# ======================================================================

plt.figure(
    figsize=(7.5, 5.0)
)

plt.plot(
    r["lead_days"],
    r["PR_AUC"],
    marker="o",
    linewidth=2,
    label="SMAP PR–AUC"
)

plt.plot(
    r["lead_days"],
    r["event_rate"],
    marker="s",
    linestyle="--",
    linewidth=1.5,
    label="Event-rate baseline"
)

plt.xlabel(
    "Lead window (days)"
)

plt.ylabel(
    "Precision–recall AUC"
)

plt.title(
    "Precision–recall performance for strict drought development"
)

plt.xticks(
    lead_times
)

plt.legend(
    frameon=False
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        strict_dir,
        "Fig_StrictOnset_PR_AUC_28d.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ======================================================================
# 9.11 ANTECEDENT-WINDOW SENSITIVITY FIGURE
# ======================================================================

plt.figure(
    figsize=(7.5, 5.0)
)

for days in ANTECEDENT_WINDOWS:

    rtemp = strict_continuous_all[
        strict_continuous_all[
            "antecedent_days"
        ] == days
    ]

    plt.plot(
        rtemp["lead_days"],
        rtemp["ROC_AUC"],
        marker="o",
        linewidth=2,
        label=f"{days}-day antecedent"
    )

plt.axhline(
    0.5,
    linestyle=":",
    linewidth=1
)

plt.xlabel(
    "Lead window (days)"
)

plt.ylabel(
    "ROC–AUC"
)

plt.title(
    "Sensitivity to antecedent drought-free period"
)

plt.xticks(
    lead_times
)

plt.legend(
    frameon=False
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        strict_dir,
        "Fig_Antecedent_Sensitivity.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ======================================================================
# 9.12 ROC AND PR CURVES FOR PRIMARY 28-DAY ANALYSIS
# ======================================================================

for L in lead_times:

    label_col = f"lab_fix_{L}"

    d = pairs[
        pairs[
            f"strict_eligible_{PRIMARY_ANTECEDENT_DAYS}d"
        ] == 1
    ].dropna(
        subset=[
            label_col,
            "neg_z"
        ]
    )

    y = (
        d[label_col]
        .astype(int)
        .values
    )

    score = (
        d["neg_z"]
        .astype(float)
        .values
    )

    if np.unique(y).size < 2:
        continue

    # ROC
    fpr, tpr, _ = roc_curve(
        y,
        score
    )

    auc = roc_auc_score(
        y,
        score
    )

    plt.figure(
        figsize=(5.8, 5.0)
    )

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"AUC = {auc:.3f}"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle=":"
    )

    plt.xlabel(
        "False-positive rate"
    )

    plt.ylabel(
        "True-positive rate"
    )

    plt.title(
        f"ROC curve: {L}-day lead"
    )

    plt.legend(
        frameon=False
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            strict_dir,
            f"ROC_curve_{L}d.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


    # Precision-recall
    precision, recall, _ = (
        precision_recall_curve(
            y,
            score
        )
    )

    ap = average_precision_score(
        y,
        score
    )

    prevalence = y.mean()

    plt.figure(
        figsize=(5.8, 5.0)
    )

    plt.plot(
        recall,
        precision,
        linewidth=2,
        label=f"PR–AUC = {ap:.3f}"
    )

    plt.axhline(
        prevalence,
        linestyle=":",
        label=(
            f"Baseline = "
            f"{prevalence:.3f}"
        )
    )

    plt.xlabel(
        "Recall"
    )

    plt.ylabel(
        "Precision"
    )

    plt.title(
        f"Precision–recall curve: {L}-day lead"
    )

    plt.legend(
        frameon=False
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            strict_dir,
            f"PR_curve_{L}d.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ======================================================================
# 9.13 CREATE FINAL PAPER TABLE
# ======================================================================

paper_table = bootstrap_28[
    [
        "lead_days",
        "N",
        "event_rate",
        "ROC_AUC",
        "ROC_AUC_CI_low",
        "ROC_AUC_CI_high",
        "PR_AUC",
        "PR_AUC_CI_low",
        "PR_AUC_CI_high"
    ]
].copy()

paper_table[
    "ROC_AUC_95CI"
] = paper_table.apply(

    lambda x:
    (
        f"{x['ROC_AUC']:.3f} "
        f"({x['ROC_AUC_CI_low']:.3f}–"
        f"{x['ROC_AUC_CI_high']:.3f})"
    ),

    axis=1
)

paper_table[
    "PR_AUC_95CI"
] = paper_table.apply(

    lambda x:
    (
        f"{x['PR_AUC']:.3f} "
        f"({x['PR_AUC_CI_low']:.3f}–"
        f"{x['PR_AUC_CI_high']:.3f})"
    ),

    axis=1
)

paper_table_final = paper_table[
    [
        "lead_days",
        "N",
        "event_rate",
        "ROC_AUC_95CI",
        "PR_AUC_95CI"
    ]
]

paper_table_final.to_csv(
    os.path.join(
        strict_dir,
        "Table_StrictOnset_Primary_28d.csv"
    ),
    index=False
)

print(
    "\n=========================================="
)
print(
    "PRIMARY PAPER TABLE"
)
print(
    "=========================================="
)

print(
    paper_table_final
    .to_string(index=False)
)


# ======================================================================
# 9.14 SAVE ANALYSIS DATA
# ======================================================================

save_cols = [

    "county_fips",
    "date",
    "sm_z",
    "neg_z",
    "trigger_5D",
    "DSCI",
    "event"

]

save_cols += [

    f"antecedent_clear_{d}d"
    for d in ANTECEDENT_WINDOWS

]

save_cols += [

    f"strict_eligible_{d}d"
    for d in ANTECEDENT_WINDOWS

]

save_cols += [

    f"lab_fix_{L}"
    for L in lead_times

]

pairs[
    save_cols
].to_parquet(

    os.path.join(
        strict_dir,
        "strict_onset_verification_pairs.parquet"
    ),

    index=False
)


print(
    "\n=========================================="
)
print(
    "ANALYSIS COMPLETE"
)
print(
    "=========================================="
)

print(
    "\nFiles saved to:"
)

print(
    strict_dir
)

print(
    "\nMain files to inspect:"
)

for filename in [
    "Strict_onset_continuous_SMAP.csv",
    "Strict_onset_binary_trigger.csv",
    "Strict_onset_continuous_vs_binary.csv",
    "Strict_onset_28d_bootstrap_CI.csv",
    "Table_StrictOnset_Primary_28d.csv",
    "Fig_StrictOnset_ROC_AUC_28d.png",
    "Fig_StrictOnset_PR_AUC_28d.png",
    "Fig_Antecedent_Sensitivity.png"
]:
    print("  ", filename)

In [ ]:
# ================================================================
# 10. PERSISTENCE BASELINE: does SMAP add skill beyond initial DSCI?
# ================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

dsci_col = "DSCI"          # DSCI at forecast initialization
bins    = [-0.1, 0, 99, 149, 199]
blabels = ["0", "1-99", "100-149", "150-199"]

rows = []
for L in lead_times:
    lab = f"lab_fix_{L}"
    d = pairs.dropna(subset=[lab, "neg_z", dsci_col])
    d = d[d["event"] == 0].copy()                 # onset-only
    y = d[lab].astype(int).values

    row = {
        "lead_days": L,
        "N": len(d),
        "AUC_DSCI_only": roc_auc_score(y, d[dsci_col]),
        "AUC_SMAP_only": roc_auc_score(y, d["neg_z"]),
    }

    # Combined model (in-sample): DSCI + SMAP
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    X = d[[dsci_col, "neg_z"]].values
    row["AUC_DSCI_plus_SMAP"] = roc_auc_score(y, m.fit(X, y).predict_proba(X)[:, 1])
    row["AUC_gain_from_SMAP"] = row["AUC_DSCI_plus_SMAP"] - row["AUC_DSCI_only"]

    # SMAP skill among counties at a similar starting DSCI
    d["bin"] = pd.cut(d[dsci_col], bins=bins, labels=blabels)
    for b in blabels:
        s = d[d["bin"] == b]
        ok = len(s) > 0 and s[lab].nunique() == 2
        row[f"SMAP_AUC_DSCI_{b}"] = roc_auc_score(s[lab].astype(int), s["neg_z"]) if ok else np.nan
        row[f"N_{b}"] = len(s)

    rows.append(row)

persistence = pd.DataFrame(rows)
print(persistence.round(4).to_string(index=False))
persistence.to_csv(os.path.join(results_dir, "Sensitivity_persistence_baseline.csv"), index=False)

d = pairs[pairs["event"] == 0].copy()
d["bin"] = pd.cut(d["DSCI"], bins=[-0.1, 0, 99, 149, 199], labels=["0", "1-99", "100-149", "150-199"])
print(pd.DataFrame({L: d.groupby("bin", observed=True)[f"lab_fix_{L}"].sum() for L in lead_times}))